# CodeAlpha Task 4 — Sales Prediction using Python

This notebook predicts sales from advertising expenditure using multiple linear regression and analyzes the relationship between the available advertising channels and sales.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv("Advertising.csv")
print("Shape:", df.shape)
df.head()

Shape: (200, 5)


      TV  Radio  Newspaper  Sales
0  230.1   37.8       69.2   22.1
1   44.5   39.3       45.1   10.4
2   17.2   45.9       69.3    9.3
3  151.5   41.3       58.5   18.5
4  180.8   10.8       58.4   12.9

## 1. Data cleaning and preprocessing
The supplied file contains an `Unnamed: 0` identifier. It is not a meaningful predictive feature, so it is removed. The remaining modeling columns are checked and converted to numeric values.

In [3]:
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

features = ["TV", "Radio", "Newspaper"]
target = "Sales"

for column in features + [target]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

print("Missing values:")
print(df[features + [target]].isna().sum())
df = df.dropna(subset=features + [target])
print("Clean shape:", df.shape)

Missing values:
TV           0
Radio        0
Newspaper    0
Sales        0
Clean shape: (200, 4)


## 2. Exploratory analysis

In [4]:
print(df.describe().round(2))
print("\nCorrelation with Sales:")
print(df[features + [target]].corr()[target].drop(target).sort_values(ascending=False).round(3))

           TV   Radio  Newspaper   Sales
count  200.00  200.00     200.00  200.00
mean   147.04   23.26      30.55   14.02
std     85.85   14.85      21.78    5.22
min      0.70    0.00       0.30    1.60
25%     74.38    9.98      12.75   10.38
50%    149.75   22.90      25.75   12.90
75%    218.82   36.52      45.10   17.40
max    296.40   49.60     114.00   27.00

Correlation with Sales:
TV           0.782
Radio        0.576
Newspaper    0.228


In [5]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, feature in zip(axes, features):
    ax.scatter(df[feature], df[target], alpha=0.75)
    ax.set_xlabel(feature + " Advertising Spend")
    ax.set_ylabel("Sales")
    ax.set_title(feature + " vs Sales")
plt.tight_layout()
plt.show()

## 3. Train the regression model
A multiple Linear Regression model is trained using an 80/20 train-test split. `random_state=42` makes the split reproducible.

In [6]:
X = df[features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = mean_squared_error(y_test, predictions) ** 0.5
r2 = r2_score(y_test, predictions)

print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R²: {r2:.3f}")

MAE: 1.461
RMSE: 1.782
R²: 0.899


In [7]:
coefficients = pd.DataFrame({
    "Feature": features,
    "Coefficient": model.coef_
}).sort_values("Coefficient", ascending=False)
coefficients

  Feature  Coefficient
    Radio     0.189195
       TV     0.044730
Newspaper     0.002761

## 4. Model evaluation

In [8]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test, predictions, alpha=0.8)
mn = min(y_test.min(), predictions.min())
mx = max(y_test.max(), predictions.max())
plt.plot([mn, mx], [mn, mx])
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title("Actual vs Predicted Sales")
plt.tight_layout()
plt.show()

## 5. Findings
- The model achieved a test MAE of **1.461**, RMSE of **1.782**, and R² of **0.899**.
- The fitted coefficients are positive for TV, Radio and Newspaper.
- TV and Radio have larger coefficient magnitudes than Newspaper in this model.
- These are model-based associations; the dataset does not establish that changing one channel alone will cause a particular sales increase.
- The dataset contains no date, platform or target-segment columns, so this project does not claim to perform time-series forecasting or segment-level analysis.